In [ ]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt

class ReconciledMeasurementHarness:
    def __init__(self):
        self.registry = {}
        # Log hardware context for accurate profiling[cite: 1]
        self.device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

    @staticmethod
    def sync_time():
        """Ensures GPU operations finish before capturing time[cite: 1]."""
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return time.perf_counter()

    @staticmethod
    def relative_l2_error(u_pred, u_exact):
        """Calculates Relative L2 with epsilon safety to prevent zero-division[cite: 1]."""
        u_pred_flat = u_pred.flatten()
        u_exact_flat = u_exact.flatten()
        
        diff_norm = torch.linalg.norm(u_pred_flat - u_exact_flat, ord=2)
        exact_norm = torch.linalg.norm(u_exact_flat, ord=2)
        
        return (diff_norm / (exact_norm + 1e-8)).item()

    def benchmark_amortized_method(self, name, train_fn, inference_fn, u_exact_test, n_test_instances, n_train_samples=1000):
        """Profiles upfront training vs fast inference for Operator models[cite: 1]."""
        print(f"--- Benchmarking Operator: {name} (Hardware: {self.device_name}) ---")
        
        t_start_train = self.sync_time()
        train_fn() 
        train_time = self.sync_time() - t_start_train
        
        t_start_infer = self.sync_time()
        u_pred = inference_fn() 
        total_infer_time = self.sync_time() - t_start_infer
        per_instance_time = total_infer_time / n_test_instances
        
        l2_err = self.relative_l2_error(u_pred, u_exact_test)
        
        self.registry[name] = {
            "type": "amortized",
            "train_time_sec": train_time,
            "per_instance_sec": per_instance_time,
            "per_instance_std": 0.0,
            "n_train_samples": n_train_samples,
            "l2_error": l2_err
        }
        
        print(f"Accuracy (Rel L2) : {l2_err:.6f}")
        print(f"Train Time        : {train_time:.2f}s | Per-Instance: {per_instance_time:.6f}s\n")
        return self.registry[name]

    def benchmark_pinn_solver(self, name, pinn_solver_factory, u_exact_fn, sample_instances, n_runs=10):
        """Profiles instance-based optimization over multiple runs for statistical accuracy[cite: 1]."""
        print(f"--- Benchmarking PINN Solver over {n_runs} runs: {name} ---")
        
        runtimes = []
        errors = []
        loss_breakdowns = []

        for i in range(n_runs):
            instance_ic = sample_instances[i]
            exact_u = u_exact_fn(instance_ic)
            
            t_start = self.sync_time()
            u_pred, loss_dict = pinn_solver_factory(instance_ic)
            runtime = self.sync_time() - t_start
            
            err = self.relative_l2_error(u_pred, exact_u)
            
            runtimes.append(runtime)
            errors.append(err)
            loss_breakdowns.append(loss_dict)

        mean_runtime = float(np.mean(runtimes))
        std_runtime = float(np.std(runtimes))
        mean_error = float(np.mean(errors))
        
        avg_res_loss = np.mean([l.get('res_loss', 0.0) for l in loss_breakdowns])
        avg_bc_loss = np.mean([l.get('bc_loss', 0.0) for l in loss_breakdowns])
        avg_ic_loss = np.mean([l.get('ic_loss', 0.0) for l in loss_breakdowns])

        self.registry[name] = {
            "type": "instance",
            "train_time_sec": 0.0,
            "per_instance_sec": mean_runtime,
            "per_instance_std": std_runtime,
            "l2_error": mean_error,
            "loss_breakdown": {
                "L_res": avg_res_loss,
                "L_bc": avg_bc_loss,
                "L_ic": avg_ic_loss
            }
        }

        print(f"Mean Time/Instance: {mean_runtime:.2f}s ± {std_runtime:.2f}s")
        print(f"Mean Rel L2 Error : {mean_error:.6f}")
        print(f"Loss Breakdown    -> Residual: {avg_res_loss:.3e} | BC: {avg_bc_loss:.3e} | IC: {avg_ic_loss:.3e}\n")
        return self.registry[name]

    def plot_amortization(self, max_instances=1000):
        """Plots total cost against instances with logarithmic scaling and std shading[cite: 1]."""
        instances = np.arange(1, max_instances + 1)
        plt.figure(figsize=(10, 6))
        
        for name, metrics in self.registry.items():
            mean_cost = metrics["train_time_sec"] + (instances * metrics["per_instance_sec"])
            plt.plot(instances, mean_cost, label=f"{name} (Err: {metrics['l2_error']:.4f})", linewidth=2)
            
            if metrics["per_instance_std"] > 0:
                std_cost = instances * metrics["per_instance_std"]
                plt.fill_between(instances, mean_cost - std_cost, mean_cost + std_cost, alpha=0.15)
            
        plt.title(f"Amortization Analysis ({self.device_name})", fontsize=14)
        plt.xlabel("Number of Problem Instances (N)", fontsize=12)
        plt.ylabel("Total Wall-Clock Time (seconds)", fontsize=12)
        plt.yscale("log")
        plt.grid(True, which="both", ls="--", alpha=0.5)
        plt.legend(fontsize=11)
        plt.tight_layout()
        plt.savefig('figures/amortization_analysis.png')